# 3.1 · plan_ego — 자차는 어디로 가는가

앞으로 3초의 자차 경로를 예측합니다. 정답이 egomotion 센서 측정값에서 직접 나오므로 이 데이터셋에서 가장 신뢰도 높은 타깃입니다.

이 노트북은 네 가지를 확인합니다 — **어떤 원시 데이터에서**, **어떤 코드를 거쳐**, **무엇이 입력으로 들어가고**, **빌드된 파일이 그 코드와 일치하는지**. GPU 는 필요 없습니다.

In [ ]:
import os, sys, json, textwrap
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..")))

import numpy as np
import pandas as pd

from datatools import paths

ITEMS = os.path.join(paths.COMMON_DIR, "instruct_items_tasks01_06.parquet")
pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 80)
wrap = lambda s, i="   ": textwrap.fill(str(s), 94, initial_indent=i,
                                        subsequent_indent=i)
VARIANTS = ['plan_ego_xy', 'plan_ego_control']
print("variants:", VARIANTS)

## 1. 어떤 원시 데이터에서 오는가

| 아카이브 | 주기 | 읽는 것 |
|---|---|---|
| `egomotion` | 10 Hz | x, y, qx..qw — 이것 하나로 정답이 만들어집니다 |
| `radar` | 20 / 12.7 Hz | 2초 창 20스캔. 앞차의 감속을 읽는 데 쓰입니다 |
| `camera` | 1 Hz | 2장. 신호등·정지선·커브가 운전자 의도를 정합니다 |

## 2. 어떤 코드를 거치는가

실행 순서입니다.

| 함수 | 하는 일 |
|---|---|
| `ego_frame(ego)` | 속도, 가속도, 요, 요레이트 파생 |
| `ego_waypoints(derived, t_s)` | t+1/2/3초 위치를 현재 자차 좌표계로 회전 변환 |
| `ego_controls(derived, t_s)` | 같은 궤적을 속도·요레이트로 |
| `ego_action(derived, t_s)` | CoT 용 기동 분류 |

전부 `datatools/frame_objects.py` 와 `datatools/geometry.py` 에 있습니다.

In [ ]:
import inspect
from datatools import frame_objects as F
for name in ['ego_frame', 'ego_waypoints', 'ego_controls', 'ego_action']:
    fn = getattr(F, name, None)
    if fn is None:
        from datatools import geometry as G
        fn = getattr(G, name, None)
    if fn is None or not callable(fn):
        print(f'{name}: (모듈 함수 아님)'); continue
    doc = (inspect.getdoc(fn) or '').split(chr(10))[0]
    print(f'{name:34s} {doc[:80]}')

## 3. 입력으로 무엇이 들어가는가

**비전 2장 (1 fps) · 레이더 20스캔 (2초, 10 Hz) · ego 20샘플**

입력 창은 태스크마다 다릅니다. 로더의 `WINDOWS` 표가 그것을 정하고, 레이더는 항상 20스캔이라 창의 길이가 바뀌면 샘플링 속도가 따라 바뀝니다 — 인코더 입력 모양은 변하지 않습니다.

In [ ]:
from training.instruct_data import WINDOWS, INSTANT_TASKS, WINDOW_TASKS
for v in VARIANTS:
    for name in (v, v + "_cot"):
        if name in WINDOWS:
            secs, hz, frames = WINDOWS[name]
            print(f"{name:26s} 창 {secs}초 · 레이더 {hz} Hz × 20스캔 · 비전 {frames}장")
        elif name in INSTANT_TASKS:
            print(f"{name:26s} 순간 — 비전 1장 · 레이더 20스캔/1초 · ego 1")
        else:
            print(f"{name:26s} 클립 전체 — 비전 20장 · 레이더 20스캔/20초")

## 4. 실제 아이템

빌드된 파일에서 그대로 꺼냅니다.

In [ ]:
built = pd.read_parquet(ITEMS)
for v in VARIANTS:
    sub = built[built.task == v]
    if sub.empty:
        print(f"{v}: 파일에 없음"); continue
    r = sub.iloc[0]
    print("=" * 96)
    print(f"{v}   clip {r.clip_id[:8]}  frame {r.frame} (t={r.frame-1}s)  split {r.split}")
    print("Q:"); print(wrap(r.prompt))
    print("A:"); print(wrap(r.target))

## 5. CoT — 근거가 답을 만드는가

`_cot` 변형은 `{"rationale": ..., "answer": ...}` 입니다. **근거를 따라가면 답이 나와야** 합니다. 나오지 않으면 그 사슬은 잘못된 것이고, 보상을 걸면 모델이 그 잘못된 사슬을 배웁니다.

In [ ]:
for v in VARIANTS:
    name = v + "_cot"
    sub = built[built.task == name] if 'built' in dir() else None
    if sub is None or sub.empty:
        continue
    r = sub.iloc[0]
    d = json.loads(r.target)
    print("=" * 96); print(name)
    print("R:"); print(wrap(d["rationale"]))
    print("A:"); print(wrap(d["answer"]))

## 6. 보상

평가 채점기에서 유도했습니다. 정답을 그대로 넣으면 1.0 이 나와야 하고, 내용을 망가뜨리면 떨어져야 합니다.

In [ ]:
import re
from training.task_scorers import reward_for

def wreck(text):
    """형식은 두고 숫자만 2배로."""
    return re.sub(r"\d+(?:\.\d+)?",
                  lambda m: str(round(float(m.group()) * 2, 1)), text)

rows = []
for v in VARIANTS:
    for name in (v, v + "_cot"):
        fn = reward_for(name)
        if fn is None:
            continue
        sub = built[built.task == name] if 'built' in dir() else None
        if sub is None or sub.empty:
            continue
        t = sub.iloc[0].target
        rows.append({"task": name, "reward": fn.__name__,
                     "정답": round(fn(t, t), 3),
                     "숫자 2배": round(fn(wreck(t), t), 3)})
pd.DataFrame(rows)

## 7. 데이터 양

`val` 은 `train` 에 합쳐져 있습니다 — 클립 분할이 train 86,607 / val 54,163 / test 37,121 인데, 모델 선택은 `test` 에서 하므로 검증용 3분의 1이 쓰이지 않고 있었습니다.

In [ ]:
from collections import Counter
from training.instruct_data import load_items
names = [v for v in VARIANTS] + [v + "_cot" for v in VARIANTS]
rows = []
for split in ("train", "test"):
    c = Counter(i["task"] for i in load_items(tuple(names), split))
    for n in names:
        rows.append({"task": n, "split": split, "items": c.get(n, 0)})
pd.DataFrame(rows).pivot(index="task", columns="split", values="items")

## 8. 이 태스크에서 내린 결정과 근거

**등속 외삽으로는 부족**

현재 속도 × 시간으로 외삽하면 오차 중앙값 1.09 m 이고 보상의 반점 기준(1 m) 안에 드는 것이 47.9% 뿐입니다. 감속과 조향을 예측해야 하고, 그건 장면에 달려 있습니다.

**앵커 2초 간격**

1초 간격이면 인접 정답 차이가 중앙값 0.72 m 로 반점 기준보다 작아 같은 문항의 반복이 됩니다. 2초면 1.42 m 로 넘어섭니다.

**t=17 이 마지막**

18+3=21초는 클립 밖입니다.

**근거가 레이더가 아닌 유일한 태스크**

내가 어디로 갈지는 내 속도와 조향이 정합니다. 레이더 개입이 여기까지 영향을 준다면 그것은 레이더 능력이 아니라는 신호입니다.